In [18]:
import json
import re
from pathlib import Path

import fitz  # PyMuPDF

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
pdf_path = project_root / "source" / "verifikacija_softvera.pdf"
pages_dir = out_dir / "pages" 
pages_dir.mkdir(parents=True, exist_ok=True)

FIRST_N_PAGES = 188
print(pdf_path, pdf_path.exists())

/home/anja/Desktop/MU/projekat/source/verifikacija_softvera.pdf True


In [6]:
pip install pymupdf 

Note: you may need to restart the kernel to use updated packages.


In [7]:
import sys
!{sys.executable} -m pip install "PyMuPDF==1.24.14"


In [9]:
WATERMARK_TEXT = "Elektronska verzĳa (2026)"
WATERMARK_SIZE_THRESHOLD = 30.0
HEADING_SIZE_THRESHOLD = 12.0
HEADER_BAND_Y = 45.0

NUMBERED_HEADING_RE = re.compile(r"^(\d+(?:\.\d+)*)\s+(.*\S)\s*$")
PAGE_NUM_RE = re.compile(r"\b(\d{1,4})\b")
CONTROL_CHARS_RE = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")

In [13]:
from typing import Optional, Tuple


def _block_text(block: dict) -> str:

    lines = []
    for line in block.get("lines", []):
        spans = "".join(s["text"] for s in line["spans"])
        lines.append(spans)
    text = "\n".join(lines).strip()
    return CONTROL_CHARS_RE.sub("", text)


def _block_max_size(block: dict) -> float:
    sizes = [s["size"] for line in block.get("lines", []) for s in line["spans"]]
    return max(sizes) if sizes else 0.0


def _heading_level(text: str) -> Optional[Tuple[int, str]]:
    match = NUMBERED_HEADING_RE.match(text.replace("\n", " "))
    if not match:
        return None
    number, title = match.groups()
    level = number.count(".") + 1
    return level, f"{number} {title}"

In [14]:
def extract_page(page: fitz.Page) -> dict:
    raw = page.get_text("dict")
    body_blocks: list[tuple[float, float, str, float]] = []
    header_text = None

    for block in raw.get("blocks", []):
        if block.get("type", 0) != 0:
            continue  # preskacemo slike
        text = _block_text(block)
        if not text:
            continue
        max_size = _block_max_size(block)
        x0, y0 = block["bbox"][0], block["bbox"][1]

        if text.startswith(WATERMARK_TEXT) or max_size >= WATERMARK_SIZE_THRESHOLD:
            continue  #onaj znak na strani

        if y0 < HEADER_BAND_Y:
            header_text = text.replace("\n", " ").strip()
            continue  

        body_blocks.append((y0, x0, text, max_size))

    body_blocks.sort(key=lambda b: (round(b[0] / 4), b[1]))

    printed_page = None
    section_ref = None
    if header_text:
        digits = PAGE_NUM_RE.findall(header_text)
        if digits:
            printed_page = int(digits[0])
        section_ref = PAGE_NUM_RE.sub("", header_text).strip(" .") or None

    md_lines: list[str] = []
    current_heading = None
    for _, _, text, size in body_blocks:
        heading = _heading_level(text) if size >= HEADING_SIZE_THRESHOLD else None
        if heading:
            level, heading_text = heading
            current_heading = heading_text
            md_lines.append(f"{'#' * min(level + 1, 6)} {heading_text}")
        else:
            md_lines.append(text)

    body_text = "\n\n".join(md_lines).strip()

    return {
        "printed_page": printed_page,
        "section_ref": section_ref,
        "heading": current_heading,
        "text": body_text,
    }


In [19]:
doc = fitz.open(pdf_path)
n_pages = min(FIRST_N_PAGES, doc.page_count)

jsonl_path = out_dir / "pages.jsonl"
md_path = out_dir / "verifikacija_softvera.md"

with jsonl_path.open("w", encoding="utf-8") as jf, md_path.open("w", encoding="utf-8") as mf:
    mf.write("# Verifikacija softvera (elektronska verzija, 2026)\n\n")
    for i in range(n_pages):
        pdf_page = i + 1
        record = extract_page(doc[i])
        record["pdf_page"] = pdf_page

        (pages_dir / f"page_{pdf_page:04d}.txt").write_text(record["text"], encoding="utf-8")
        jf.write(json.dumps(record, ensure_ascii=False) + "\n")

        if record["text"]:
            page_label = record["printed_page"] if record["printed_page"] is not None else pdf_page
            mf.write(f"\n<!-- pdf_page={pdf_page} printed_page={page_label} -->\n\n")
            mf.write(record["text"] + "\n")

print(f"Ekstrahovano {n_pages} strana -> {pages_dir}, {jsonl_path}, {md_path}")


Ekstrahovano 188 strana -> /home/anja/Desktop/MU/projekat/data/processed/pages, /home/anja/Desktop/MU/projekat/data/processed/pages.jsonl, /home/anja/Desktop/MU/projekat/data/processed/verifikacija_softvera.md


In [20]:
records = [json.loads(line) for line in jsonl_path.read_text(encoding="utf-8").splitlines() if line]
non_empty = [r for r in records if r["text"]]
print(f"Ukupno strana: {len(records)}, sa tekstom: {len(non_empty)}")

sample = next(r for r in records if r["pdf_page"] == 100)
print("--- primer strane 100 ---")
print(sample["text"][:600])

Ukupno strana: 188, sa tekstom: 182
--- primer strane 100 ---
omogućavaju da se testovi ponavljaju automatski ili u
serĳama na više uređaja odjednom, čime se povećava
ponovljivost i pouzdanost testiranja.

Instalacioni testovi se nekada sprovode i u saradnji sa
krajnjim korisnicima, kako bi se osiguralo da instalacĳa
teče bez grešaka i da sistem nakon instalacĳe ispravno
funkcioniše. Poseban akcenat stavlja se na detekcĳu
eventualnih uticaja okruženja na funkcionalne i nefunk-
cionalne karakteristike sistema, kao što su performanse,
sigurnost ili kompatibilnost. Ovo testiranje ima za cilj da
potvrdi da je softver spreman za rad u stvarnom okruže-
nju kor
